In [1]:
from vllm import LLM, SamplingParams

INFO 06-13 07:23:38 [__init__.py:224] Automatically detected platform rocm.


In [2]:
TARGET_MODEL = "google/medgemma-1.5-4b-it"

In [3]:
one_shot_input = (
    "A 48-year-old female treated with Tofacitinib for rheumatoid arthritis presented with "
    "a painful, unilateral vesicular eruption along the T4 dermatome, confirming herpes zoster reactivation."
)

In [4]:
one_shot_output = (
    '{"seriousness": true, "meddra_terms": ["Herpes zoster", "Rash vesicular", "Pain"], '
    '"labeling_status": ["Herpes zoster: Listed", "Rash vesicular: Listed", "Pain: Listed"], '
    '"naranjo_score": 7, "rationale": "JAK inhibition suppressing cell-mediated viral surveillance '
    'pathways, allowing latent varicella-zoster virus reactivation."}'
)

In [5]:
new_case = (
    "A 54-year-old male patient with a history of rheumatoid arthritis was prescribed "
    "Adalimumab (Humira) 40mg subcutaneously every other week. Two weeks after the third dose, "
    "the patient developed a severe high fever (39.5°C), dry cough, and acute dyspnea. "
    "Chest X-ray revealed bilateral interstitial infiltrates. The patient was admitted to the ICU "
    "with suspected drug-induced interstitial lung disease. Adalimumab was immediately discontinued, "
    "and high-dose intravenous methylprednisolone was initiated. Symptoms resolved over 10 days."
)

In [6]:
prompt_template = (
    f"System: You are a PV extraction engine. Output ONLY a single-line flattened JSON.\n\n"
    f"Example Input: {one_shot_input}\n"
    f"Example Output: {one_shot_output}\n\n"
    f"User: Input Text: {new_case}\n"
    f"Assistant:"
)

In [7]:
prompt_template

'System: You are a PV extraction engine. Output ONLY a single-line flattened JSON.\n\nExample Input: A 48-year-old female treated with Tofacitinib for rheumatoid arthritis presented with a painful, unilateral vesicular eruption along the T4 dermatome, confirming herpes zoster reactivation.\nExample Output: {"seriousness": true, "meddra_terms": ["Herpes zoster", "Rash vesicular", "Pain"], "labeling_status": ["Herpes zoster: Listed", "Rash vesicular: Listed", "Pain: Listed"], "naranjo_score": 7, "rationale": "JAK inhibition suppressing cell-mediated viral surveillance pathways, allowing latent varicella-zoster virus reactivation."}\n\nUser: Input Text: A 54-year-old male patient with a history of rheumatoid arthritis was prescribed Adalimumab (Humira) 40mg subcutaneously every other week. Two weeks after the third dose, the patient developed a severe high fever (39.5°C), dry cough, and acute dyspnea. Chest X-ray revealed bilateral interstitial infiltrates. The patient was admitted to the

In [8]:
sampling_params = SamplingParams(
    temperature=0.1,  # Kept low to enforce deterministic structure try
    max_tokens=300
)

In [ ]:
try:
    # gpu_memory_utilization is set to 0.70 to maximize context cache space on a single laptop card safely
    llm = LLM(
        model=TARGET_MODEL, 
        gpu_memory_utilization=0.40, 
        trust_remote_code=True,
        max_model_len=1024
    )
    
    outputs = llm.generate([prompt_template], sampling_params)
    generated_text = outputs
    
    print(f"\n🔴 RAW UNSTRUCTURED RESPONSE FROM {TARGET_MODEL}:")
    print("=" * 70)
    print(generated_text)
    print("=" * 70)

except Exception as e:
    print(f"\n❌ Execution Error for {TARGET_MODEL}: {e}")

INFO 06-13 07:15:25 [utils.py:239] non-default args: {'trust_remote_code': True, 'max_model_len': 1024, 'gpu_memory_utilization': 0.4, 'disable_log_stats': True, 'model': 'google/medgemma-1.5-4b-it'}
INFO 06-13 07:15:25 [model.py:653] Resolved architecture: Gemma3ForConditionalGeneration


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 06-13 07:15:25 [model.py:1714] Using max model len 1024


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 06-13 07:15:27 [scheduler.py:225] Chunked prefill is enabled with max_num_batched_tokens=16384.
WARNING 06-13 07:15:31 [__init__.py:2879] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 06-13 07:15:33 [__init__.py:224] Automatically detected platform rocm.
(EngineCore_DP0 pid=12496) INFO 06-13 07:15:35 [core.py:727] Waiting for init message from front-end.
(EngineCore_DP0 pid=12496) INFO 06-13 07:15:35 [core.py:94] Initializing a V1 LLM engine (v0.11.0rc2.dev424+g045b396d0) with config: model='google/medgemma-1.5-4b-it', speculative_config=None, tokenizer='google/medgemma-1.5-4b-it', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=1024, download_dir=None, load_format=auto, tensor_parallel_

In [ ]:
import os

# AMD Notebook Environment Overrides
os.environ["HSA_OVERRIDE_GFX_VERSION"] = "11.0.0"
os.environ["VLLM_USE_MODELSCOPE"] = "0"

from vllm import LLM, SamplingParams

TARGET_MODEL = "google/medgemma-1.5-4b-it"

# 1. Initialize the engine with mandatory bfloat16 for AMD compatibility
print(f"🚀 Initializing {TARGET_MODEL} in strict bfloat16 mode...")
llm = LLM(
    model=TARGET_MODEL,
    gpu_memory_utilization=0.45,       # Keeps VRAM headroom clear for your notebook OS
    max_model_len=1024,
    dtype="bfloat16",                  # CRITICAL: Stops float16 precision underflow/NaN errors on ROCm
    trust_remote_code=True,
    enforce_eager=True,                # 🔥 CRITICAL: Bypasses the HIP graph segmentation faults on AMD
    distributed_executor_backend="mp"  # 🧠 The modern replacement for worker_use_ray=False
)

# 2. Define the structured scenario
system_instruction = "Extract PV data into a single-line flattened JSON string."
sample_narrative = (
    "A 54-year-old male patient with a history of rheumatoid arthritis was prescribed "
    "Adalimumab (Humira) 40mg subcutaneously every other week. Two weeks after the third dose, "
    "the patient developed a severe high fever (39.5°C), dry cough, and acute dyspnea."
)

# 3. Format using the formal Open-AI style schema
# MedGemma maps these roles perfectly to its underlying training tokens
messages = [
    {
        "role": "user", 
        "content": f"{system_instruction}\n\nInput Text: {sample_narrative}"
    }
]

# 4. Use vLLM's internal tokenizer to build the precise <start_of_turn> wrappers
tokenizer = llm.get_tokenizer()
formatted_prompt = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True
)

# 5. Execute with strict deterministic sampling
sampling_params = SamplingParams(
    temperature=0.0, 
    max_tokens=300
)

print("\n[System] Sending structured prompt to MedGemma core...")
try:
    outputs = llm.generate([formatted_prompt], sampling_params)
    generated_text = outputs
    
    print("\n🟢 MEDGEMMA RESPONSE:")
    print("=" * 80)
    print(generated_text)
    print("=" * 80)

except Exception as e:
    print(f"Execution failed: {e}")

🚀 Initializing google/medgemma-1.5-4b-it in strict bfloat16 mode...
INFO 06-13 07:23:43 [utils.py:239] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 1024, 'distributed_executor_backend': 'mp', 'gpu_memory_utilization': 0.45, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'google/medgemma-1.5-4b-it'}
INFO 06-13 07:23:43 [model.py:653] Resolved architecture: Gemma3ForConditionalGeneration


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 06-13 07:23:43 [model.py:1714] Using max model len 1024


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 06-13 07:23:45 [scheduler.py:225] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 06-13 07:23:45 [vllm.py:358] Cudagraph is disabled under eager mode
WARNING 06-13 07:23:49 [__init__.py:2879] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 06-13 07:23:52 [__init__.py:224] Automatically detected platform rocm.
(EngineCore_DP0 pid=12933) INFO 06-13 07:23:53 [core.py:727] Waiting for init message from front-end.
(EngineCore_DP0 pid=12933) INFO 06-13 07:23:53 [core.py:94] Initializing a V1 LLM engine (v0.11.0rc2.dev424+g045b396d0) with config: model='google/medgemma-1.5-4b-it', speculative_config=None, tokenizer='google/medgemma-1.5-4b-it', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16